# Exploratory Data Analysis and Feature-Group Inspection for HIGGS Classification

**Author:** Aditya Rana (data & classical baseline lead) | **Project:** Higgs Event Lab | **Module:** MID-02

This notebook provides the exploratory data analysis (EDA) and feature-group inspection for the HIGGS benchmark (Baldi et al., 2014). It verifies data hygiene, empirical class balance, discrete versus continuous variable characteristics, and comparative distributions across signal and background events.

> **Scientific boundary notice:** All collision events analysed here were produced via Monte Carlo simulation tools. Predictions represent binary classification scores on simulated benchmark events and do not claim physical observation or discovery.

## 1. Environment and Data Loading

**Question:** How do we ingest a representative, validated sample of the HIGGS dataset without exhausting system memory or risking data leakage?

**Why it matters:** The raw HIGGS dataset contains 11 million rows (approx 2.8 GB compressed and 8 GB uncompressed). We must use our tested chunked loading pipeline from  to stream a manageable sample while strictly isolating target labels from the feature attributes.

**Explanation:** We import standard scientific Python packages alongside our project module . We load fifty thousand events from the processed subset, enforcing deterministic schema separation.

**Expected output:** Confirmation of loaded rows and distinct feature counts.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

from higgs_event_lab.data import load_higgs_sample
from higgs_event_lab.features import (
    ALL_COLUMNS,
    ALL_FEATURES,
    FEATURE_GROUPS,
    HIGH_LEVEL_FEATURES,
    LOW_LEVEL_FEATURES,
    TARGET_COLUMN,
    get_feature_columns,
)

data_path = Path("data/processed/higgs_sample_250k.csv.gz")
if not data_path.exists():
    data_path = Path("data/raw/HIGGS.csv.gz")

sample_rows = 50000
batch = load_higgs_sample(data_path, n_rows=sample_rows, feature_group="all")
X = batch.X
y = batch.y

print("Feature matrix shape:", X.shape)
print("Target label vector shape:", y.shape)
print("Low-level feature count:", len(LOW_LEVEL_FEATURES))
print("High-level feature count:", len(HIGH_LEVEL_FEATURES))

## 2. Dataset Schema and Target Label Semantics

**Question:** What do the binary target labels represent, and how are the input columns partitioned across physical subsystems?

**Why it matters:** Precise semantic clarity prevents confusion between the 2012 Higgs boson discovery and this exotic-Higgs benchmark. In this benchmark,  denotes simulated signal events with hypothetical heavy particles, while  denotes standard model top-pair background events with identical decay products.

**Explanation:** We inspect the exact column breakdown and verify that the target column  is never present inside the feature DataFrame.

**Expected output:** Feature group breakdown confirming 21 low-level kinematic variables and 7 high-level invariant masses.

In [ ]:
assert TARGET_COLUMN not in X.columns
print("Target column name:", TARGET_COLUMN)
print("Unique label values:", np.unique(y))

group_summary = pd.DataFrame([
    {"Group": "Target", "Count": 1, "Variables": TARGET_COLUMN},
    {"Group": "Low-level kinematics", "Count": len(LOW_LEVEL_FEATURES), "Variables": f"{LOW_LEVEL_FEATURES[0]} ... {LOW_LEVEL_FEATURES[-1]}"},
    {"Group": "High-level invariant masses", "Count": len(HIGH_LEVEL_FEATURES), "Variables": f"{HIGH_LEVEL_FEATURES[0]} ... {HIGH_LEVEL_FEATURES[-1]}"},
])
print(group_summary.to_string(index=False))

## 3. Data Integrity and Missing Value Verification

**Question:** Are there any null, missing, NaN, or non-finite values present in the loaded HIGGS sample?

**Why it matters:** Missing values in collider reconstruction can indicate corrupted sensor streams or failed kinematic fits. Downstream algorithms like Logistic Regression and compact MLPs will produce invalid gradients or throw exceptions if non-finite entries exist.

**Explanation:** We run exhaustive checks across all 28 feature columns and the label array for nulls, infinite values, or improper data types.

**Expected output:** Zero missing values, zero infinite values, and uniform float64 numerical representations.

In [ ]:
null_counts = X.isnull().sum()
total_nulls = int(null_counts.sum())
infinite_counts = int(np.isinf(X.to_numpy()).sum())
label_nulls = int(pd.Series(y).isnull().sum())

print("Total missing values across all features:", total_nulls)
print("Total infinite values across all features:", infinite_counts)
print("Total missing values in target array:", label_nulls)
print("All columns numeric dtype:", all(np.issubdtype(dt, np.number) for dt in X.dtypes))

## 4. Class Balance and Signal Fraction Analysis

**Question:** What is the exact class distribution between signal and background events in this empirical sample?

**Why it matters:** Many machine learning benchmarks suffer from extreme class imbalance requiring synthetic resampling or asymmetric cost weighting. Previous project documentation noted that class balance was unverified (BUG-014); we compute the exact empirical signal fraction to ground the project facts.

**Explanation:** We compute the counts, percentages, and imbalance ratio between signal (class 1) and background (class 0) collision events.

**Expected output:** Empirical count and percentage for both classes confirming approximate balance.

In [ ]:
label_counts = pd.Series(y).value_counts().sort_index()
signal_count = int(label_counts.get(1, 0))
bkg_count = int(label_counts.get(0, 0))
total_count = len(y)
signal_fraction = signal_count / total_count

print(f"Background (class 0): {bkg_count:,} ({bkg_count / total_count:.2%})")
print(f"Signal (class 1):     {signal_count:,} ({signal_count / total_count:.2%})")
print(f"Empirical signal fraction: {signal_fraction:.4f}")
print(f"Imbalance ratio (Signal / Background): {signal_count / bkg_count:.4f}")

## 5. Discrete Feature Verification: Jet B-tagging Properties

**Question:** Do the jet b-tagging variables exhibit continuous or discrete values, and what physical meaning does this convey?

**Why it matters:** Physics documentation (BUG-014 and section 4 of the dataset guide) indicated that b-tagging might be discrete rather than continuous. Machine learning preprocessing must treat discrete features appropriately rather than assuming smooth Gaussian variations.

**Explanation:** We examine unique value frequencies and zero-fraction metrics across the four b-tagging columns (, , , ).

**Expected output:** Table showing unique value counts and the predominance of discrete values.

In [ ]:
btag_cols = ["jet_1_btag", "jet_2_btag", "jet_3_btag", "jet_4_btag"]
btag_stats = []
for col in btag_cols:
    vals = X[col].value_counts(normalize=True).head(5)
    zero_pct = float((X[col] == 0.0).mean())
    btag_stats.append({
        "Feature": col,
        "Unique count": X[col].nunique(),
        "Zero fraction": f"{zero_pct:.2%}",
        "Top value": f"{vals.index[0]:.2f} ({vals.iloc[0]:.1%})",
    })
print(pd.DataFrame(btag_stats).to_string(index=False))

## 6. Kinematic Feature Distributions: Signal versus Background

**Question:** How do low-level particle kinematic variables (transverse momenta, pseudorapidity, azimuthal angles) differ between signal and background?

**Why it matters:** Understanding the physical separation in 1D projections highlights why linear models struggle on raw kinematics alone. For instance, azimuthal angles are uniformly distributed due to detector cylindrical symmetry, whereas transverse momenta exhibit heavy tails.

**Explanation:** We compute summary statistics (mean and standard deviation) split by signal and background across representative low-level features.

**Expected output:** Comparative tabular statistics showing subtle shifts in transverse momenta and near identical angular spreads.

In [ ]:
kinematic_subset = ["lepton_pt", "lepton_eta", "missing_energy_magnitude", "jet_1_pt"]
kin_comparison = []
for col in kinematic_subset:
    sig_mean = float(X.loc[y == 1, col].mean())
    sig_std = float(X.loc[y == 1, col].std())
    bkg_mean = float(X.loc[y == 0, col].mean())
    bkg_std = float(X.loc[y == 0, col].std())
    kin_comparison.append({
        "Feature": col,
        "Signal Mean ± Std": f"{sig_mean:.3f} ± {sig_std:.3f}",
        "Bkg Mean ± Std": f"{bkg_mean:.3f} ± {bkg_std:.3f}",
        "Abs Diff": f"{abs(sig_mean - bkg_mean):.3f}",
    })
print(pd.DataFrame(kin_comparison).to_string(index=False))

## 7. High-Level Derived Invariant Mass Distributions

**Question:** Do physicist-engineered invariant masses provide clearer discrimination between exotic Higgs decay signals and background top-quark processes?

**Why it matters:** Invariant masses represent composite particle resonance signatures. For example,  reconstructs candidate Higgs boson candidates decaying into bottom quark pairs, while  reconstructs top decay chains.

**Explanation:** We compare signal and background distributions for key invariant masses (, , , ), calculating separation metrics between the two class distributions.

**Expected output:** Statistical comparisons demonstrating noticeable distributional shifts in invariant masses for signal events compared to background.

In [ ]:
mass_cols = list(HIGH_LEVEL_FEATURES)[:4]
mass_comparison = []
for col in mass_cols:
    sig_mean = float(X.loc[y == 1, col].mean())
    sig_std = float(X.loc[y == 1, col].std())
    bkg_mean = float(X.loc[y == 0, col].mean())
    bkg_std = float(X.loc[y == 0, col].std())
    diff = abs(sig_mean - bkg_mean)
    mass_comparison.append({
        "High-level Feature": col,
        "Signal Mean ± Std": f"{sig_mean:.3f} ± {sig_std:.3f}",
        "Bkg Mean ± Std": f"{bkg_mean:.3f} ± {bkg_std:.3f}",
        "Separation": f"{diff / ((sig_std + bkg_std) / 2):.3f}",
    })
print(pd.DataFrame(mass_comparison).to_string(index=False))

## 8. Correlation Structure Across Feature Groups

**Question:** How strongly correlated are low-level kinematic variables compared to high-level derived invariant mass features?

**Why it matters:** High collinearity among features impacts linear model stability and coefficient interpretability. Tree models and regularized neural networks handle collinearity differently, making it crucial to map the cross-feature correlation structure.

**Explanation:** We compute the Pearson correlation matrix for the high-level features and analyze the strongest correlations between high-level and low-level variables.

**Expected output:** Summary of highest correlated feature pairs.

In [ ]:
corr_matrix = X.corr(method="pearson")
high_level_corr = corr_matrix.loc[list(HIGH_LEVEL_FEATURES), list(HIGH_LEVEL_FEATURES)]

pairs = []
for i in range(len(HIGH_LEVEL_FEATURES)):
    for j in range(i + 1, len(HIGH_LEVEL_FEATURES)):
        f1, f2 = HIGH_LEVEL_FEATURES[i], HIGH_LEVEL_FEATURES[j]
        val = high_level_corr.loc[f1, f2]
        pairs.append((f1, f2, abs(val), val))

pairs.sort(key=lambda item: item[2], reverse=True)
top_corr_df = pd.DataFrame([
    {"Feature 1": p[0], "Feature 2": p[1], "Correlation": f"{p[3]:+.3f}"}
    for p in pairs[:5]
])
print("Top high-level feature correlations:")
print(top_corr_df.to_string(index=False))

## 9. Synthesis and Implications for Machine Learning Modeling

**Question:** What key modeling decisions and preprocessing guidelines emerge from this exploratory analysis for downstream classifiers?

**Why it matters:** The exploratory findings directly motivate the experimental design for both the classical Logistic Regression baseline and the compact Multi-Layer Perceptron.

**Explanation:** We summarize four verified empirical takeaways: first, continuous features exhibit varying scales requiring StandardScaler; second, b-tag features are zero-inflated discrete indicators; third, target class balance is roughly 53/47 eliminating the need for resampling; fourth, high-level invariant masses contain concentrated discriminatory signal that simple linear models cannot reconstruct from low-level variables alone.

**Expected output:** A concise structured summary of empirical modeling principles.

In [ ]:
summary_insights = pd.DataFrame([
    {"Aspect": "Class Balance", "Finding": f"{signal_fraction:.1%} signal / {1 - signal_fraction:.1%} background", "Implication": "Standard cross-entropy loss without class re-weighting"},
    {"Aspect": "Data Quality", "Finding": "0 missing or non-finite values across all 28 features", "Implication": "No imputation needed; raw tensors ready for scaling"},
    {"Aspect": "Feature Scales", "Finding": "pT and invariant masses have long right tails", "Implication": "StandardScaler in Pipeline required for Logistic Regression"},
    {"Aspect": "Feature Groups", "Finding": "High-level masses show superior 1D separation", "Implication": "Benchmark low_level vs high_level vs all to test representation learning"},
])
print(summary_insights.to_string(index=False))